# PILOT 2주

# TASK 0. 데이터 전처리에 앞서

# 이전 과제에서의 문제점

In [63]:
# last_season(시즌) 컬럼이 2023 이상인 데이터만
df_recent = df_players_bf[df_players_bf['last_season'] >= 2023]

In [64]:
# 선수들이 뛰고 있는 리그도 '유럽 5대 빅리그' 선수만 남기겠습니다
# GB1: 잉글랜드 프리미어리그, ES1: 스페인 라리가, IT1: 이탈리아 세리에A,
# L1: 독일 분데스리가, FR1: 프랑스 리그앙
big_5_leagues = ['GB1', 'ES1', 'IT1', 'L1', 'FR1']
df_players = df_recent[df_recent['current_club_domestic_competition_id'].isin(big_5_leagues)]

In [65]:
# 데이터 수 확인
print(len(df_players))

4689


In [66]:
df_players.shape

(4689, 23)

# 다른 데이터셋 확인해보기

players를 확인했는데, appearances나 valuations도 확인해보겠습니다

In [67]:
df_appearances_before = pd.read_csv('/content/appearances.csv.gz')
df_raw = df_appearances_before.copy()

In [68]:
df_appearances_before.head()

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45
3,2234418_73333,2234418,73333,1274,6646,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90


In [69]:
df_player_valuations_before = pd.read_csv('/content/player_valuations.csv.gz')
df_raw = df_player_valuations_before.copy()

In [70]:
df_player_valuations_before.head()

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1
3,6893,2003-12-15,900000,Galatasaray,984,GB1
4,12359,2004-03-11,250000,RC Lens B,8152,NaN


# 마찬가지로, 정말 방대한 데이터 양입니다

이번에도 필터링을 해보겠습니다.

과제에서 했던 것처럼 필터링한 고유 id를 활용할 예정입니다

In [71]:
# 앞서 필터링한 선수들의 고유 ID(player_id) 목록을 추출하고
valid_player_ids = df_players['player_id'].unique()
print(f"✅ 필터링한 선수 수: {len(valid_player_ids):,}명\n")

✅ 필터링한 선수 수: 4,689명



In [72]:
# 필터링 선수들의 기록만 남기기
df_appearances = df_appearances_before[df_appearances_before['player_id'].isin(valid_player_ids)].copy()

In [73]:
df_appearances.head()

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
39,2234404_55735,2234404,55735,660,46,2012-07-09,Henrikh Mkhitaryan,UKRS,0,0,0,0,90
222,2222963_206717,2222963,206717,6994,667,2012-07-14,Eduard Sobol,UKR1,0,0,0,0,90
344,2222959_55735,2222959,55735,660,46,2012-07-15,Henrikh Mkhitaryan,UKR1,0,0,2,2,90
430,2224730_48015,2224730,48015,3426,162,2012-07-15,Lukas Hradecky,DK1,0,0,0,0,90
469,2224731_107775,2224731,107775,2414,89,2012-07-16,Frederik Rönnow,DK1,0,0,0,0,90


In [74]:
df_appearances.shape

(449285, 13)

In [75]:
# 마찬가지로 player_valuations (과거 몸값 히스토리) 데이터 필터링
# 앞서 필터링했던 선수들의 기록만 남기기
df_player_valuations = df_player_valuations_before[df_player_valuations_before['player_id'].isin(valid_player_ids)].copy()

In [76]:
df_player_valuations.head()

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
5,3333,2004-04-10,7500000,Newcastle United,399,GB1
6,12029,2004-04-10,1500000,Valencia CF,347,FR1
10,14086,2004-04-10,50000,Watford FC,1010,GB1
11,14555,2004-04-10,400000,Leeds United,399,GB1
12,15452,2004-04-10,800000,Getafe CF,6182,NaN


In [77]:
df_player_valuations.shape

(41048, 6)

# 최종

흩어져 있던 데이터를 하나의 완벽한 표로 병합하는 게 좋아보입니다

다행히 제가 앞에서 모든 데이터가 player_id라는 공통된 열쇠(Key)를 가지고 있기 때문에 당연히 하나로 합칠 수 있을 것 같은데요.

players 데이터는 **선수 1명당 1줄**입니다.

appearances 데이터는 **선수 1명당 여러 줄(출전한 경기 수만큼)**입니다.

그래서 appearances 데이터를 선수별로 묶어서(groupby) '총 득점', '총 어시스트' 등으로 합산한 뒤에 합치려고 합니다.

In [78]:
# 1. appearances 데이터를 선수별로 묶어서 스탯 총합(sum) 구하기
# player_id를 기준으로 출전 시간, 골, 어시스트를 모두 더해주기
player_stats = df_appearances.groupby('player_id')[['minutes_played', 'goals', 'assists']].sum().reset_index()

# 2. 선수 데이터(df_players)와 스탯 데이터(player_stats) 병합
# 'left'로 왼쪽 데이터(df_players)를 기준으로 뼈대를 잡고 합칩니다
df = pd.merge(df_players, player_stats, on='player_id', how='left')

# 3. 부상 관련 경기는 NaN으로 나왔고 이를 0으로 채워줍니다.
df[['minutes_played', 'goals', 'assists']] = df[['minutes_played', 'goals', 'assists']].fillna(0)

# 4. 나이 파생 변수 만들기
# 생년월일을 날짜 데이터로 변환한 뒤, 현재 연도(2026)에서 태어난 연도를 뺍니다.
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'])
df['age'] = 2026 - df['date_of_birth'].dt.year

# 5. 최종 병합된 데이터 확인
cols_to_show = ['name', 'age', 'current_club_domestic_competition_id', 'position',
                'minutes_played', 'goals', 'assists', 'market_value_in_eur']

print(f"최종 병합된 데이터 크기: {df.shape}")
display(df[cols_to_show].head(10))

최종 병합된 데이터 크기: (4689, 27)


,name,age,current_club_domestic_competition_id,position,minutes_played,goals,assists,market_value_in_eur
0,James Milner,40,GB1,Midfield,"26,456",39,70,"750,000"
1,Jonas Hofmann,34,L1,Midfield,"21,843",70,83,"2,000,000"
2,Pepe Reina,44,ES1,Goalkeeper,"30,675",0,2,"600,000"
3,Philipp Pentke,41,L1,Goalkeeper,"1,110",0,0,"300,000"
4,Ludovic Butelle,43,FR1,Goalkeeper,"13,533",0,1,"100,000"
5,Daley Blind,36,ES1,Defender,"40,502",23,26,"1,400,000"
6,Alessio Cragno,32,IT1,Goalkeeper,"16,308",0,0,"1,500,000"
7,Andy Lonergan,43,GB1,Goalkeeper,0,0,0,"100,000"
8,Ashley Young,41,GB1,Defender,"26,590",18,44,"400,000"
9,Scott Carson,41,GB1,Goalkeeper,"3,062",0,0,"250,000"


# TASK 1. 데이터 전처리


## 결측치 확인

In [79]:
df.isnull().sum()

,0
player_id,0
first_name,160
last_name,0
name,0
last_season,0
current_club_id,0
player_code,0
country_of_birth,186
city_of_birth,186
country_of_citizenship,0


In [80]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

if len(missing_values) > 0:
    print(missing_values)

agent_name                     1240
contract_expiration_date        549
highest_market_value_in_eur     437
market_value_in_eur             437
height_in_cm                    233
foot                            190
city_of_birth                   186
country_of_birth                186
first_name                      160
date_of_birth                     3
age                               3
sub_position                      1
dtype: int64


In [81]:
# 원본 데이터를 보존하기 위해 복사본 만들기
df_cleaned = df.copy()

# 전략 1. 예측할 '정답(몸값)'이 없는 데이터 삭제
df_cleaned = df_cleaned.dropna(subset=['market_value_in_eur'])

In [82]:
# 불필요하고 결측치가 너무 많은 컬럼 제외시키기
# 에이전트 이름, 계약 만료일, 역대 최고 몸값 제외
# 태어난 국가와 도시도 중요한 요소는 아니니 제외
# name 칸이 따로 있으니 first name도 지워주기
cols_to_drop = ['agent_name', 'city_of_birth', 'country_of_birth',
                'first_name', 'highest_market_value_in_eur', 'contract_expiration_date']
df_cleaned = df_cleaned.drop(columns=cols_to_drop, errors='ignore')

In [83]:
missing_values = df_cleaned.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

if len(missing_values) > 0:
    print(missing_values)

height_in_cm     121
foot              97
date_of_birth      1
sub_position       1
age                1
dtype: int64


결측치인 키와 주발을 어떻게 처리해야 할지가 고민입니다

[방법 1] 그냥 없앤다 (그렇게 많은 양의 데이터를 처리하고 있지는 않다)

1%도 안되는 양의 수치이기 때문이다

In [84]:
print(df_cleaned.shape)

(4252, 21)


In [85]:
# [방법 2] 유의미한 결측치일 수도 있으니 그 값을 평균치로 때려넣는다

# 키는 평균 키로, 주발은 'right'로 채우는 방식입니다

# 제미나이한테 코드를 짜달라고 한 다음

# 결측치에 해당하는 선수들의 몸값을 봤는데

In [86]:
import matplotlib.pyplot as plt
# 1. 비교를 위해 '몸값'은 적혀 있는 선수들만 남깁니다.
df_test = df.dropna(subset=['market_value_in_eur']).copy()

# 2. '키(height_in_cm)'가 비어있는지 여부를 나타내는 새로운 변수를 만듭니다.
# True면 빈칸(결측치), False면 키가 적혀있는 정상 데이터입니다.
df_test['is_missing_profile'] = df_test['height_in_cm'].isnull()

# 3. 두 그룹의 평균, 최댓값 등 기초 통계량 비교
print("=== 📊 프로필 결측 여부에 따른 몸값 통계 비교 ===")
# 지수표현식(e+06) 방지 및 보기 쉽게 출력
pd.options.display.float_format = '{:,.0f}'.format
display(df_test.groupby('is_missing_profile')['market_value_in_eur'].describe())

plt.show()

=== 📊 프로필 결측 여부에 따른 몸값 통계 비교 ===


,count,mean,std,min,25%,50%,75%,max
is_missing_profile,,,,,,,,
False,"4,131","8,384,058","15,641,545","10,000","800,000","2,500,000","9,000,000","200,000,000"
True,121,"221,240","224,005","10,000","100,000","150,000","250,000","1,000,000"


max가 1,000,000인 것으로 보아, 없어도 되겠다는 생각이 드네요

그냥 삭제시켜버리겠습니다

In [87]:
# 키, 주발 등 빈칸이 하나라도 있는 선수 가차없이 삭제!
cols_to_check = ['height_in_cm', 'foot', 'age', 'sub_position']
df_cleaned = df_cleaned.dropna(subset=cols_to_check)

In [88]:
# 이상치는 실제 축구 시장을 보여줄 수 있는 데이터이기에
# 삭제하지 않도록 하겠습니다

In [89]:
# 최종 확인
df_cleaned.isnull().sum()

,0
player_id,0
last_name,0
name,0
last_season,0
current_club_id,0
player_code,0
country_of_citizenship,0
date_of_birth,0
sub_position,0
position,0


In [90]:
# 중복 데이터 확인
df_cleaned.duplicated().sum()

np.int64(0)

### 추가 전처리: 데이터 인코딩 및 스케일링 (변환)

머신러닝 모델의 학습 효율과 예측 성능을 극대화하기 위해 다음과 같은 추가 전처리를 수행했습니다.

**1. 데이터 인코딩: 원-핫 인코딩 (One-Hot Encoding)**
* **대상:** `position` (포지션), `foot` (주발)
* **이유 및 방법:** 머신러닝 알고리즘은 문자형 데이터를 직접 연산할 수 없습니다. 따라서 텍스트로 된 범주형 변수들을 컴퓨터가 이해할 수 있도록 0과 1로 이루어진 가변수(Dummy Variable)로 변환하는 원-핫 인코딩을 적용했습니다. (다중공선성 문제를 방지하기 위해 `drop_first=True` 옵션을 적용하여 기준이 되는 첫 번째 범주는 제외했습니다.)

**2. 데이터 스케일링/변환: 로그 변환 (Log Transformation)**
* **대상:** `market_value_in_eur` (타겟 변수 - 이적시장 몸값)
* **이유 및 방법:** 앞선 이상치 확인 과정에서 타겟 데이터가 우측으로 극단적으로 치우친(Right-skewed) 분포를 띠고 있음을 확인했습니다. 이처럼 단위가 지나치게 크고 편차가 심한 데이터를 그대로 학습시키면 모델의 오차가 기하급수적으로 커지게 됩니다.
* 따라서 타겟 변수에 `np.log1p()`를 적용하여 **로그 변환(Log Transformation)**을 수행했습니다. 이를 통해 극단적인 이상치의 영향을 줄이고, 데이터 분포를 정규분포(Normal Distribution)에 가깝게 압축하여 모델이 안정적으로 학습할 수 있는 환경을 구축했습니다.

In [91]:
import numpy as np
import pandas as pd

# 복사본
df_ml_ready = df_cleaned.copy()

# attack_points 컬럼이 df_ml_ready에 없으므로 다시 생성합니다.
df_ml_ready['attack_points'] = df_ml_ready['goals'] + df_ml_ready['assists']

# 1. 데이터 인코딩
# 컴퓨터가 읽지 못하는 글자 데이터(포지션, 주발)를 0과 1로 변환
categorical_cols = ['position', 'foot']
df_ml_ready = pd.get_dummies(df_ml_ready, columns=categorical_cols, drop_first=True)

# 2. 데이터 스케일링/변환
# 몸값 데이터가 우측으로 꼬리가 긴 형태(슈퍼스타 극단치)이므로 로그 변환을 적용
df_ml_ready['market_value_log'] = np.log1p(df_ml_ready['market_value_in_eur'])

# 결과 확인 최종 머신러닝용 데이터 엿보기
# 이름 등 머신러닝에 넣지 않을 문자열은 제외하고, 변환된 컬럼들 위주로 출력
display(df_ml_ready[['name', 'market_value_in_eur', 'market_value_log', 'age', 'attack_points'] +
                    [col for col in df_ml_ready.columns if 'position_' in col or 'foot_' in col]].head())

,name,market_value_in_eur,market_value_log,age,attack_points,position_Defender,position_Goalkeeper,position_Midfield,foot_left,foot_right
0,James Milner,"750,000",14,40,109,False,False,True,False,True
1,Jonas Hofmann,"2,000,000",15,34,153,False,False,True,False,True
2,Pepe Reina,"600,000",13,44,2,False,True,False,False,True
3,Philipp Pentke,"300,000",13,41,0,False,True,False,False,True
4,Ludovic Butelle,"100,000",12,43,1,False,True,False,True,False


# TASK 2. 피쳐 엔지니어링


# TASK 2. 피쳐 엔지니어링

머신러닝 모델이 선수의 시장 가치를 더 잘 학습할 수 있도록 기존 변수들을 결합 및 변환하여 **2개의 핵심 파생변수**를 생성할게요.

**1. 파생변수 age (만 나이)**
* **결합/변환 방식:** `date_of_birth` (생년월일) 변수와 '현재 날짜'를 연산하여 만 나이를 정수형(Integer)으로 계산
* **생성 이유 및 활용:** 원본 데이터의 생년월일(YYYY-MM-DD) 형식은 날짜 데이터라 인공지능이 그 자체로 크고 작음을 연산하기 어렵습니다. 이적시장에서 선수의 몸값을 결정하는 가장 치명적인 요인인 '에이징 커브'와 '유망주 프리미엄'을 모델이 직접적으로 인지하고 학습할 수 있도록, 직관적인 수치형 데이터인 `age`로 변환하여 분석에 활용했습니다.
* **코딩** 위에 만들어놨습니다.

**2. 파생변수 attack_points (누적 공격포인트)**
* **결합/변환 방식:** `goals` (골) 변수와 `assists` (어시스트) 변수를 더하여 하나의 변수로 결합했습니다.
* **생성 이유 및 활용:** 골과 어시스트를 분리해서 보는 것보다, 두 스탯을 합쳐 선수의 **'종합적인 득점 기여도'**로 묶었을 때 이적시장 몸값과의 뚜렷한 양(+)의 상관관계를 훨씬 더 강력하게 설명할 수 있음을 EDA 과정에서 확인했습니다. 공격수와 미드필더의 가치를 평가하는 가장 강력한 수치형 피쳐로 활용할 계획입니다.

In [93]:
# 새로운 변수 만들기: 공격포인트 = 골 + 어시스트
df_cleaned['attack_points'] = df_cleaned['goals'] + df_cleaned['assists']